# E1 · MultiAxial / nnU-Net — Colab hazırlığı

**Kaynak:** `handoff_k2/K2_BASLA.md`, `hazirlik_notlari_en.md`, `splits_final.json`.
Sabit **44 eğitim + 10 doğrulama**, tek T1 kanalı, 0–5 etiketleri, `3d_fullres`, yalnız **fold 0**.
Test verisi kullanılmaz. Bu defter CHARM çalıştırmaz, hazır ağ ağırlığı indirmez.

1. Hazırlanan `handoff_k2_e1.zip` dosyasını kendi Drive'ında `TMS_E1/` klasörüne yükle.
2. Colab'da bu defteri aç; GPU seç (en az 12 GB VRAM; tamamlanan deneyde L4 kullanıldı).
3. Hücreleri sırayla çalıştır. Önizlemeleri ve veri notlarını incelemeden eğitim açılmaz.
4. Önce 5 epoch kontrol; 100 epoch ön model ayrı, açık bir seçimdir.

**Varsayılan ayarlar eğitim başlatmaz.** GPU oturumunu açık bırakmak yine de birim tüketebilir.
Sonuçlar/checkpoint'ler Drive'da, eğitim verisi ve ön işleme Colab'ın yerel diskinde tutulur.
Google hesabına giriş ve Drive erişim onayı sana aittir.


In [ ]:
# Change paths if you uploaded the prepared archive elsewhere.
from pathlib import Path
import json, os, re, shutil, subprocess, sys, time

ARCHIVE = Path("/content/drive/MyDrive/TMS_E1/handoff_k2_e1.zip")
RESULTS_BASE = Path("/content/drive/MyDrive/TMS_E1/runs")
EXPERIMENT_ID = "E1_baseline_001"  # Keep unchanged when resuming the SAME experiment.
ACKNOWLEDGE_PROVISIONAL_DATA = False  # Review labels/units/NYU geometry caveats first.
RUN_5_EPOCH_PILOT = False
RUN_100_EPOCH_PRELIMINARY = False  # Enable only after reviewing pilot and budget.
RESUME_PILOT = False
RESUME_PRELIMINARY = False
CU_PER_HOUR = 0.0  # Enter this GPU session's actual displayed rate; 0 means unknown.
AVAILABLE_CU = 0.0  # Current ACCOUNT balance, not a sum of notebooks' balances.
AUGMENTATION_WORKERS = 2
assert re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,63}", EXPERIMENT_ID)
assert 1 <= AUGMENTATION_WORKERS <= 4
assert CU_PER_HOUR >= 0 and AVAILABLE_CU >= 0
WORK = Path("/content/e1_work")
TOOLS = Path("/content/e1_tools")
VENV = Path("/content/e1-venv")
PYTHON = VENV / "bin/python"
DATA = WORK / "handoff"
PREPROCESSED = WORK / "nnUNet_preprocessed"
RESULTS = RESULTS_BASE / EXPERIMENT_ID


In [ ]:
# Fingerprints of the locally audited handoff; do not change for a resumed experiment.
EXPECTED_DATASET_SHA256 = 'a3d7801021287c538ee8f3af55947b3795fe3ccf5d628f01d34942e0b4f93ae0'
EXPECTED_SPLIT_SHA256 = '9e2b37ffccff8196f9457963e3e603eabfeae6e89d8baec91ab5f989845d78b8'


## 1 · Drive bağlantısı
Yalnız kendi erişimin olan arşivi kullan. Drive'daki orijinal MRI dosyaları değiştirilmez.


In [ ]:
assert sys.platform == "linux", "Bu defteri Google Colab'ın Linux oturumunda aç."
from google.colab import drive
drive.mount("/content/drive")
assert os.path.ismount("/content/drive"), "Kalıcı Drive bağlantısı kurulamadı."
assert ARCHIVE.is_file(), f"Hazırlanan ZIP bulunamadı: {ARCHIVE}"
WORK.mkdir(parents=True, exist_ok=True)
TOOLS.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
assert shutil.disk_usage("/content").free > 20 * 1024**3, "Kurulum/veri için en az 20 GiB boş yer ayır."


## 2 · Ayrı, sürümü sabitlenmiş eğitim ortamı
Colab çekirdeğinin PyTorch'unu değiştirmek yerine Python 3.11 sanal ortamı kullanılır.
Linux/CUDA bağımlılık çözümlemesi hazırlanmıştır; gerçek GPU çalışması aşağıda kontrol edilir.
İlk kurulum birkaç GB indirebilir. Gömülü yardımcılar bu proje kaynaklarından üretilmiştir;
ZIP'teki metinlerden veya başka depolardan kod çalıştırılmaz.


In [ ]:
# Embedded, versioned helpers; no private repository authentication needed.
EMBEDDED = {'audit_handoff.py': '"""Read-only technical audit of the fixed E1 handoff; never trains a model."""\n\nimport argparse\nfrom collections import Counter\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\n\nimport nibabel as nib\nimport numpy as np\n\n\nfrom colab_helpers import DATASET, require, sha256\n\nLABELS = {"background": 0, "skin": 1, "skull": 2, "csf": 3, "gray_matter": 4, "white_matter": 5}\n\n\ndef audit(root):\n    root = Path(root).resolve()\n    dataset = root / "nnUNet_raw" / DATASET\n    for path in (root / "test_holdout", dataset / "imagesTs", dataset / "labelsTs"):\n        require(not path.exists(), f"Hold-out/test directory must stay outside this training package: {path.name}")\n    metadata = json.loads((dataset / "dataset.json").read_text())\n    require(metadata.get("labels") == LABELS, "Unexpected six-class label contract.")\n    require(metadata.get("channel_names") == {"0": "T1"}, "Expected one T1 channel.")\n    require(metadata.get("file_ending") == ".nii.gz", "Expected .nii.gz images.")\n    folds = json.loads((root / "splits_final.json").read_text())\n    require(len(folds) == 1 and set(folds[0]) == {"train", "val"}, "Expected exactly one fixed train/val fold.")\n    train, val = folds[0]["train"], folds[0]["val"]\n    require(len(train) == 44 and len(val) == 10, "Expected 44 train and 10 validation subjects.")\n    require(all(isinstance(case, str) and re.fullmatch(r"[A-Za-z0-9_-]+", case) for case in train + val), "Invalid case ID.")\n    require(len(set(train + val)) == 54, "Duplicate or overlapping train/validation subjects.")\n    cases = set(train + val)\n    require(metadata.get("numTraining") == len(cases), "dataset.json numTraining disagrees with the split.")\n    images = {p.name for p in (dataset / "imagesTr").iterdir()}\n    labels = {p.name for p in (dataset / "labelsTr").iterdir()}\n    require(images == {f"{case}_0000.nii.gz" for case in cases}, "imagesTr contains missing, extra, or incorrectly named inputs.")\n    require(labels == {f"{case}.nii.gz" for case in cases}, "labelsTr contains missing, extra, or incorrectly named labels.")\n    report = {"dataset": DATASET, "train_count": 44, "validation_count": 10,\n              "test_included": False, "labels": LABELS, "cases": [], "files": {},\n              "warnings": [\n                  "Label semantic mapping is inferred in the handoff and still needs review.",\n                  "Removed-face ignore masks are absent; this is a preliminary tissue experiment, not validated face reconstruction.",\n                  "Handoff reports orthonormalized NYU affines; original transforms/manifests are not included. Physical-space accuracy is not established.",\n                  "Exact duplicates are checked, but subject identity and external test separation still rely on K1 provenance."\n              ]}\n    hashes = {}\n    for case in sorted(cases):\n        image_path = dataset / "imagesTr" / f"{case}_0000.nii.gz"\n        label_path = dataset / "labelsTr" / f"{case}.nii.gz"\n        image, label = nib.load(image_path), nib.load(label_path)\n        require(isinstance(image, (nib.Nifti1Image, nib.Nifti2Image)) and isinstance(label, (nib.Nifti1Image, nib.Nifti2Image)), f"{case}: expected NIfTI images.")\n        require(len(image.shape) == 3 and image.shape == label.shape, f"{case}: image/label shape mismatch or non-3D data.")\n        require(0 < np.prod(image.shape, dtype=np.int64) <= 128_000_000, f"{case}: unsupported volume size.")\n        require(np.isfinite(image.affine).all() and abs(np.linalg.det(image.affine[:3, :3])) > 1e-8, f"{case}: invalid affine.")\n        require(np.allclose(image.affine, label.affine, atol=1e-4, rtol=0), f"{case}: image/label affines disagree.")\n        units = (image.header.get_xyzt_units()[0], label.header.get_xyzt_units()[0])\n        require(all(unit in ("mm", "unknown") for unit in units), f"{case}: spatial units need explicit conversion before nnU-Net.")\n        if "unknown" in units:\n            report["warnings"].append(f"{case}: NIfTI spatial units are unspecified; 1 mm is a handoff claim, not verified metadata.")\n        data = image.get_fdata(dtype=np.float32, caching="unchanged")\n        target = np.asanyarray(label.dataobj)\n        require(np.isfinite(data).all() and np.isfinite(target).all(), f"{case}: non-finite values.")\n        require(float(data.max()) > float(data.min()), f"{case}: constant image.")\n        ids, counts = np.unique(target, return_counts=True)\n        require(set(ids.tolist()) <= set(LABELS.values()), f"{case}: unexpected or non-integer label values: {ids.tolist()}")\n        require(np.any(ids > 0), f"{case}: empty foreground labels.")\n        voxel_hash = hashlib.sha256(np.ascontiguousarray(data).tobytes()).hexdigest()\n        require(voxel_hash not in hashes, f"{case}: duplicate image voxel data with {hashes.get(voxel_hash)}.")\n        hashes[voxel_hash] = case\n        spacing = nib.affines.voxel_sizes(image.affine)\n        directions = image.affine[:3, :3] / spacing\n        require(np.allclose(directions.T @ directions, np.eye(3), atol=1e-4), f"{case}: sheared affine is incompatible with this ITK training handoff.")\n        row = {"id": case, "split": "train" if case in train else "val", "shape": list(image.shape),\n               "voxel_sizes_in_header_units": spacing.tolist(), "axis_codes": "".join(nib.aff2axcodes(image.affine)),\n               "header_spatial_units": list(units),\n               "label_counts": {str(int(key)): int(value) for key, value in zip(ids, counts)},\n               "image_voxels_sha256": voxel_hash}\n        report["cases"].append(row)\n        for path in (image_path, label_path):\n            report["files"][path.relative_to(root).as_posix()] = sha256(path)\n        print(f"Checked {case} ({row[\'split\']})", flush=True)\n    for path in (dataset / "dataset.json", root / "splits_final.json"):\n        report["files"][path.relative_to(root).as_posix()] = sha256(path)\n    report["dataset_sha256"] = hashlib.sha256(json.dumps(report["files"], sort_keys=True).encode()).hexdigest()\n    report["split_sha256"] = sha256(root / "splits_final.json")\n    report["centers"] = {part: dict(Counter(re.sub(r"\\d+$", "", case) for case in folds[0][part])) for part in ("train", "val")}\n    report["technical_checks_passed"] = True\n    report["anatomical_or_semantic_approval"] = False\n    return report\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("handoff", type=Path)\n    parser.add_argument("--out", type=Path, required=True)\n    args = parser.parse_args()\n    result = audit(args.handoff)\n    args.out.parent.mkdir(parents=True, exist_ok=True)\n    args.out.write_text(json.dumps(result, indent=2) + "\\n")\n    print(f"Passed technical checks: 44 train / 10 val. Report: {args.out}")\n\n\nif __name__ == "__main__":\n    main()\n', 'colab_helpers.py': '"""Small reusable Colab helpers; importing this module never starts training."""\n\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport shutil\nimport signal\nimport stat\nimport subprocess\nimport tempfile\nimport time\nimport zipfile\n\nDATASET = "Dataset001_MultiAxial"\n\n\ndef require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef gpu_probe():\n    import importlib.metadata\n    import torch\n    require(importlib.metadata.version("nnunetv2") == "2.8.1", "Expected nnU-Net 2.8.1.")\n    require(torch.__version__.split("+")[0] == "2.6.0" and torch.version.cuda == "12.4", "Expected torch 2.6.0 / CUDA 12.4.")\n    require(torch.cuda.is_available(), "Choose a GPU runtime in Colab; CUDA is unavailable.")\n    props = torch.cuda.get_device_properties(0)\n    require(props.total_memory >= 12 * 1024**3, "This handoff requests a GPU with at least 12 GB VRAM.")\n    layer = torch.nn.Conv3d(1, 2, 3).cuda()\n    value = torch.randn(1, 1, 12, 12, 12, device="cuda")\n    with torch.autocast("cuda"):\n        result = layer(value).square().mean()\n    result.backward()\n    torch.cuda.synchronize()\n    del layer, value, result\n    torch.cuda.empty_cache()\n    return {"gpu": props.name, "vram_gib": props.total_memory / 1024**3,\n            "torch": torch.__version__, "cuda": torch.version.cuda,\n            "nnunetv2": importlib.metadata.version("nnunetv2"), "conv3d_backward_passed": True}\n\n\ndef pack_handoff(root, report, destination):\n    root, destination = Path(root).resolve(), Path(destination).resolve()\n    require(not destination.exists(), "Archive already exists; choose a new output filename.")\n    require(not any((parent / ".git").exists() for parent in destination.parents), "Keep MRI archives outside Git.")\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    manifest = {"files": report["files"], "dataset_sha256": report["dataset_sha256"]}\n    partial = destination.with_suffix(destination.suffix + ".partial")\n    try:\n        with zipfile.ZipFile(partial, "x", compression=zipfile.ZIP_STORED) as archive:\n            for relative, digest in sorted(manifest["files"].items()):\n                source = root / relative\n                require(sha256(source) == digest, f"Input changed after audit: {relative}")\n                archive.write(source, relative)\n            archive.writestr("expected_files.json", json.dumps(manifest, indent=2) + "\\n")\n        partial.rename(destination)\n    except BaseException:\n        partial.unlink(missing_ok=True)\n        raise\n    return sha256(destination)\n\n\ndef extract_handoff(archive_path, destination):\n    archive_path, destination = Path(archive_path), Path(destination).resolve()\n    archive_digest = sha256(archive_path)\n    marker = destination / ".archive_sha256"\n    if destination.exists():\n        require(marker.is_file() and marker.read_text().strip() == archive_digest,\n                "Existing extraction belongs to another archive. Choose a fresh destination.")\n        return destination\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(archive_path) as archive:\n        names = archive.namelist()\n        require(len(names) == len(set(names)), "Archive has duplicate paths.")\n        require("expected_files.json" in names and archive.getinfo("expected_files.json").file_size < 2_000_000,\n                "Use the prepared handoff_k2_e1.zip with its checksum manifest.")\n        manifest = json.loads(archive.read("expected_files.json"))\n        expected = manifest["files"]\n        require(set(names) == set(expected) | {"expected_files.json"}, "Archive contains unexpected or missing entries.")\n        require(sum(info.file_size for info in archive.infolist()) <= 20_000_000_000, "Archive is too large.")\n        for info in archive.infolist():\n            path = PurePosixPath(info.filename)\n            require(not path.is_absolute() and ".." not in path.parts and "\\\\" not in info.filename,\n                    "Unsafe archive path.")\n            require(not stat.S_ISLNK(info.external_attr >> 16) and not info.is_dir(), "Archive links/directories are not supported.")\n            allowed = (info.filename in ("splits_final.json", "expected_files.json", f"nnUNet_raw/{DATASET}/dataset.json")\n                       or (path.parent.as_posix() in (f"nnUNet_raw/{DATASET}/imagesTr", f"nnUNet_raw/{DATASET}/labelsTr")\n                           and path.name.endswith(".nii.gz")))\n            require(allowed, "Archive contains files outside the E1 dataset.")\n        with tempfile.TemporaryDirectory(prefix="e1-extract-", dir=destination.parent) as folder:\n            temporary = Path(folder)\n            for info in archive.infolist():\n                output = temporary / info.filename\n                output.parent.mkdir(parents=True, exist_ok=True)\n                with archive.open(info) as source, output.open("xb") as target:\n                    shutil.copyfileobj(source, target, length=1024 * 1024)\n                if info.filename in expected:\n                    require(sha256(output) == expected[info.filename], f"Archive checksum mismatch: {info.filename}")\n            (temporary / ".archive_sha256").write_text(archive_digest + "\\n")\n            temporary.rename(destination)\n    return destination\n\n\ndef verify_manifest(root, report):\n    manifest = json.loads((Path(root) / "expected_files.json").read_text())\n    require(report["files"] == manifest["files"] and report["dataset_sha256"] == manifest["dataset_sha256"],\n            "Extracted data differs from the audited handoff manifest.")\n\n\ndef install_split(root, preprocessed):\n    source = Path(root) / "splits_final.json"\n    target = Path(preprocessed) / DATASET / "splits_final.json"\n    target.parent.mkdir(parents=True, exist_ok=True)\n    if target.exists():\n        require(json.loads(target.read_text()) == json.loads(source.read_text()), "Existing split differs; refusing to overwrite it.")\n    else:\n        shutil.copyfile(source, target)\n    return sha256(target)\n\n\ndef training_command(executable, trainer, fold_directory, resume=False):\n    require(trainer in ("nnUNetTrainer_E1Smoke", "nnUNetTrainer_E1Preliminary"), "Unexpected E1 trainer.")\n    folder = Path(fold_directory)\n    if resume:\n        checkpoints = [folder / f"checkpoint_{name}.pth" for name in ("final", "latest", "best")]\n        checkpoint = next((path for path in checkpoints if path.is_file()), None)\n        require(checkpoint is not None, "Resume requested but no checkpoint exists; refusing to silently restart.")\n        require(not (folder / "validation/summary.json").is_file(), "Run already has completed validation; inspect results or use a new experiment ID.")\n        require(zipfile.is_zipfile(checkpoint), "Checkpoint is incomplete or uses an unexpected format.")\n        with zipfile.ZipFile(checkpoint) as archive:\n            require(archive.testzip() is None, "Checkpoint archive failed its integrity check.")\n    else:\n        require(not folder.exists(), "Run directory exists. Use resume or a new experiment ID; no overwrite allowed.")\n    return [str(executable), "1", "3d_fullres", "0", "-tr", trainer, "-device", "cuda"] + (["--c"] if resume else [])\n\n\ndef record_run(path, contract):\n    path = Path(path)\n    if path.exists():\n        require(json.loads(path.read_text()) == contract, "Experiment contract changed; use a new experiment ID.")\n    else:\n        path.parent.mkdir(parents=True, exist_ok=True)\n        path.write_text(json.dumps(contract, indent=2) + "\\n")\n\n\ndef run_logged(command, log_path, env=None):\n    """Tail a durable log; stop owned process group when the notebook is interrupted."""\n    log_path = Path(log_path)\n    log_path.parent.mkdir(parents=True, exist_ok=True)\n    started = time.monotonic()\n    child_env = dict(os.environ if env is None else env)\n    child_env["PYTHONUNBUFFERED"] = "1"\n    with log_path.open("x", buffering=1) as output:\n        process = subprocess.Popen(command, stdout=output, stderr=subprocess.STDOUT,\n                                   stdin=subprocess.DEVNULL, env=child_env, start_new_session=True)\n        try:\n            with log_path.open(errors="replace") as reader:\n                while True:\n                    text = reader.read()\n                    if text:\n                        print(text, end="", flush=True)\n                    code = process.poll()\n                    if code is not None:\n                        print(reader.read(), end="", flush=True)\n                        if code != 0:\n                            raise subprocess.CalledProcessError(code, command)\n                        return {"elapsed_seconds": time.monotonic() - started, "exit_code": code}\n                    time.sleep(.3)\n        finally:\n            # Colab is Linux; group ownership prevents orphaned augmentation workers.\n            try:\n                os.killpg(process.pid, signal.SIGTERM)\n            except ProcessLookupError:\n                pass\n            if process.poll() is None:\n                try:\n                    process.wait(timeout=10)\n                except subprocess.TimeoutExpired:\n                    os.killpg(process.pid, signal.SIGKILL)\n                    process.wait()\n            try:\n                os.killpg(process.pid, signal.SIGKILL)\n            except ProcessLookupError:\n                pass\n', 'preview_handoff.py': '"""Create training-only voxel-plane previews; never modifies source NIfTI files."""\n\nimport argparse\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nfrom matplotlib.colors import ListedColormap\nfrom matplotlib.patches import Patch\nimport nibabel as nib\nimport numpy as np\n\nfrom audit_handoff import DATASET, LABELS\n\n\ndef create_preview(root, destination):\n    import json\n    root, destination = Path(root), Path(destination)\n    train = json.loads((root / "splits_final.json").read_text())[0]["train"]\n    selected = [next(case for case in train if case.startswith(center)) for center in ("GU", "NC", "NYU")]\n    selected += [case for case in ("NYU003", "NYU006", "NYU007", "NYU008") if case in train and case not in selected]\n    colors = [(0, 0, 0, 0), (.99, .65, .52, .45), (1, .94, .70, .5), (.4, .64, 1, .5), (.65, .32, .8, .5), (.9, .9, .9, .5)]\n    figure, axes = plt.subplots(len(selected), 3, figsize=(12, len(selected) * 3.5), squeeze=False)\n    dataset = root / "nnUNet_raw" / DATASET\n    for row, case in enumerate(selected):\n        image = nib.as_closest_canonical(nib.load(dataset / "imagesTr" / f"{case}_0000.nii.gz"))\n        labels = nib.as_closest_canonical(nib.load(dataset / "labelsTr" / f"{case}.nii.gz"))\n        data = image.get_fdata(dtype=np.float32, caching="unchanged")\n        target = np.asanyarray(labels.dataobj)\n        low, high = np.percentile(data[data > 0], [.5, 99.5])\n        for column, axis in enumerate((2, 1, 0)):\n            index = data.shape[axis] // 2\n            plane = np.flipud(np.take(data, index, axis).T)\n            mask = np.flipud(np.take(target, index, axis).T)\n            ax = axes[row, column]\n            ax.imshow(plane, cmap="gray", vmin=low, vmax=high)\n            ax.imshow(mask, cmap=ListedColormap(colors), vmin=0, vmax=5, interpolation="nearest")\n            ax.set_title(f"{case} · axis {axis}, voxel {index}")\n            ax.axis("off")\n    figure.suptitle("E1 provisional label mapping — voxel-plane QC, not physical-space validation", fontsize=13)\n    figure.legend(handles=[Patch(color=colors[label][:3], label=f"{label}: {name}") for name, label in LABELS.items() if label],\n                  loc="lower center", ncol=5)\n    figure.tight_layout(rect=(0, .025, 1, .97))\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    figure.savefig(destination, dpi=130)\n    plt.close(figure)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("handoff", type=Path)\n    parser.add_argument("--out", type=Path, required=True)\n    args = parser.parse_args()\n    create_preview(args.handoff, args.out)\n', 'trainers.py': '"""nnU-Net 2.8.1 E1 variants: seed initialization and save each epoch."""\n\nimport random\nimport numpy as np\nimport torch\n\nfrom nnunetv2.training.nnUNetTrainer.variants.training_length.nnUNetTrainer_Xepochs import (\n    nnUNetTrainer_5epochs, nnUNetTrainer_100epochs,\n)\n\n\nclass nnUNetTrainer_E1Smoke(nnUNetTrainer_5epochs):\n    def __init__(self, plans, configuration, fold, dataset_json, device=torch.device("cuda")):\n        random.seed(42)\n        np.random.seed(42)\n        torch.manual_seed(42)\n        super().__init__(plans, configuration, fold, dataset_json, device)\n        self.save_every = 1\n\n\nclass nnUNetTrainer_E1Preliminary(nnUNetTrainer_100epochs):\n    def __init__(self, plans, configuration, fold, dataset_json, device=torch.device("cuda")):\n        random.seed(42)\n        np.random.seed(42)\n        torch.manual_seed(42)\n        super().__init__(plans, configuration, fold, dataset_json, device)\n        self.save_every = 1\n', 'requirements-colab.lock.txt': '# This file was autogenerated by uv via the following command:\n#    uv pip compile tools/e1_training/requirements-colab.txt --python-version 3.11 --python-platform x86_64-unknown-linux-gnu --torch-backend cu124 --output-file /private/tmp/e1-linux-requirements.txt\nacvl-utils==0.2.7\n    # via nnunetv2\nannotated-types==0.8.0\n    # via pydantic\nanyio==4.16.0\n    # via\n    #   httpx\n    #   httpx2\nbatchgenerators==0.25.6\n    # via\n    #   acvl-utils\n    #   batchgeneratorsv2\n    #   nnunetv2\nbatchgeneratorsv2==0.3.7\n    # via nnunetv2\nblosc2==4.14.1\n    # via\n    #   acvl-utils\n    #   nnunetv2\ncertifi==2026.10.23\n    # via\n    #   httpcore\n    #   httpx\n    #   requests\ncharset-normalizer==3.5.2\n    # via requests\nclick==8.5.0\n    # via huggingface-hub\ncloudpickle==3.1.2\n    # via joblib\nconnected-components-3d==4.1.1\n    # via acvl-utils\ncontourpy==1.3.3\n    # via matplotlib\ncycler==0.12.1\n    # via matplotlib\ndynamic-network-architectures==0.4.4\n    # via nnunetv2\neinops==0.8.2\n    # via\n    #   dynamic-network-architectures\n    #   nnunetv2\nfilelock==4.2.7\n    # via\n    #   huggingface-hub\n    #   torch\nfonttools==4.68.0\n    # via matplotlib\nfsspec==2026.9.0\n    # via\n    #   huggingface-hub\n    #   torch\ngraphviz==0.21\n    # via nnunetv2\nh11==0.16.0\n    # via\n    #   httpcore\n    #   httpcore2\nh2==4.4.1\n    # via httpx\nhf-xet==1.8.0\n    # via huggingface-hub\nhpack==4.2.0\n    # via h2\nhttpcore==1.0.9\n    # via httpx\nhttpcore2==2.16.0\n    # via httpx2\nhttpx==0.28.1\n    # via blosc2\nhttpx2==2.16.0\n    # via huggingface-hub\nhuggingface-hub==2.4.0\n    # via timm\nhyperframe==6.1.0\n    # via h2\nidna==3.20\n    # via\n    #   anyio\n    #   httpx\n    #   httpx2\n    #   requests\nimagecodecs==2026.1.14\n    # via nnunetv2\nimageio==2.38.1\n    # via scikit-image\nimportlib-resources==7.1.0\n    # via nibabel\njinja2==3.1.6\n    # via torch\njoblib==1.6.0\n    # via scikit-learn\nkiwisolver==1.5.1\n    # via matplotlib\nlazy-loader==0.6\n    # via scikit-image\nmarkdown-it-py==4.2.0\n    # via rich\nmarkupsafe==3.0.4\n    # via jinja2\nmatplotlib==3.11.2\n    # via\n    #   nnunetv2\n    #   seaborn\nmdurl==0.1.2\n    # via markdown-it-py\nmpmath==1.3.0\n    # via sympy\nmsgpack==1.2.4\n    # via blosc2\nnarwhals==2.28.0\n    # via scikit-learn\nndindex==1.10.1\n    # via blosc2\nnetworkx==3.6.1\n    # via\n    #   scikit-image\n    #   torch\nnibabel==5.4.2\n    # via\n    #   -r tools/e1_training/requirements-colab.txt\n    #   nnunetv2\nnnunetv2==2.8.1\n    # via -r tools/e1_training/requirements-colab.txt\nnumexpr==2.14.2\n    # via blosc2\nnumpy==1.26.4\n    # via\n    #   -r tools/e1_training/requirements-colab.txt\n    #   acvl-utils\n    #   batchgenerators\n    #   batchgeneratorsv2\n    #   blosc2\n    #   connected-components-3d\n    #   contourpy\n    #   dynamic-network-architectures\n    #   imagecodecs\n    #   imageio\n    #   matplotlib\n    #   nibabel\n    #   nnunetv2\n    #   numexpr\n    #   pandas\n    #   scikit-image\n    #   scikit-learn\n    #   scipy\n    #   seaborn\n    #   tifffile\n    #   torchvision\nnvidia-cublas-cu12==12.4.5.8\n    # via\n    #   nvidia-cudnn-cu12\n    #   nvidia-cusolver-cu12\n    #   torch\nnvidia-cuda-cupti-cu12==12.4.127\n    # via torch\nnvidia-cuda-nvrtc-cu12==12.4.127\n    # via torch\nnvidia-cuda-runtime-cu12==12.4.127\n    # via torch\nnvidia-cudnn-cu12==9.1.0.70\n    # via torch\nnvidia-cufft-cu12==11.2.1.3\n    # via torch\nnvidia-curand-cu12==10.3.5.147\n    # via torch\nnvidia-cusolver-cu12==11.6.1.9\n    # via torch\nnvidia-cusparse-cu12==12.3.1.170\n    # via\n    #   nvidia-cusolver-cu12\n    #   torch\nnvidia-cusparselt-cu12==0.6.2\n    # via torch\nnvidia-nccl-cu12==2.21.5\n    # via torch\nnvidia-nvjitlink-cu12==12.4.127\n    # via\n    #   nvidia-cusolver-cu12\n    #   nvidia-cusparse-cu12\n    #   torch\nnvidia-nvtx-cu12==12.4.127\n    # via torch\npackaging==26.3\n    # via\n    #   huggingface-hub\n    #   lazy-loader\n    #   matplotlib\n    #   nibabel\n    #   scikit-image\npandas==3.0.6\n    # via\n    #   batchgenerators\n    #   nnunetv2\n    #   seaborn\npillow==13.0.0\n    # via\n    #   batchgenerators\n    #   imageio\n    #   matplotlib\n    #   scikit-image\n    #   torchvision\npydantic==2.14.2\n    # via blosc2\npydantic-core==2.50.2\n    # via pydantic\npygments==2.21.0\n    # via rich\npyparsing==3.3.3\n    # via matplotlib\npython-dateutil==4.0.0\n    # via\n    #   matplotlib\n    #   pandas\npyyaml==6.0.3\n    # via\n    #   huggingface-hub\n    #   timm\n    #   yacs\nrequests==2.34.2\n    # via nnunetv2\nrich==15.0.0\n    # via blosc2\nsafetensors==0.8.0\n    # via timm\nscikit-image==0.26.0\n    # via\n    #   acvl-utils\n    #   batchgenerators\n    #   nnunetv2\nscikit-learn==1.9.1\n    # via\n    #   batchgenerators\n    #   nnunetv2\nscipy==1.17.1\n    # via\n    #   batchgenerators\n    #   nnunetv2\n    #   scikit-image\n    #   scikit-learn\nseaborn==0.13.2\n    # via nnunetv2\nsimpleitk==2.5.7\n    # via\n    #   acvl-utils\n    #   nnunetv2\nsympy==1.13.1\n    # via torch\nthreadpoolctl==3.7.0\n    # via\n    #   batchgenerators\n    #   blosc2\n    #   scikit-learn\ntifffile==2026.3.3\n    # via\n    #   nnunetv2\n    #   scikit-image\ntimm==1.0.22\n    # via dynamic-network-architectures\ntorch==2.6.0+cu124\n    # via\n    #   -r tools/e1_training/requirements-colab.txt\n    #   acvl-utils\n    #   batchgeneratorsv2\n    #   dynamic-network-architectures\n    #   nnunetv2\n    #   timm\n    #   torchvision\ntorchvision==0.21.0+cu124\n    # via timm\ntqdm==4.70.1\n    # via\n    #   acvl-utils\n    #   huggingface-hub\n    #   nnunetv2\ntriton==3.2.0\n    # via torch\ntruststore==0.10.5\n    # via\n    #   httpcore2\n    #   httpx2\ntyping-extensions==4.16.0\n    # via\n    #   anyio\n    #   httpx2\n    #   huggingface-hub\n    #   nibabel\n    #   pydantic\n    #   pydantic-core\n    #   torch\n    #   typing-inspection\ntyping-inspection==0.4.4\n    # via pydantic\ntzdata==2026.6\n    # via python-dateutil\nurllib3==2.8.0\n    # via requests\nyacs==0.1.8\n    # via nnunetv2\n'}
import hashlib
for name, source in EMBEDDED.items():
    (TOOLS / name).write_text(source)
sys.path.insert(0, str(TOOLS))
from colab_helpers import (extract_handoff, verify_manifest, install_split, training_command, record_run, run_logged, require, sha256, DATASET)
LOCK_SHA256 = sha256(TOOLS / 'requirements-colab.lock.txt')
TRAINER_SHA256 = sha256(TOOLS / 'trainers.py')


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.11.8"], check=True)
uv = [sys.executable, "-m", "uv"]
if not PYTHON.is_file():
    subprocess.run(uv + ["venv", str(VENV), "--python", "3.11"], check=True)
subprocess.run(uv + ["pip", "install", "--python", str(PYTHON), "--torch-backend", "cu124",
                     "-r", str(TOOLS / "requirements-colab.lock.txt")], check=True)
subprocess.run(uv + ["pip", "check", "--python", str(PYTHON)], check=True)
trainer_path = Path(subprocess.check_output([str(PYTHON), "-c",
    "from pathlib import Path; import nnunetv2; print(Path(nnunetv2.__file__).parent / 'training/nnUNetTrainer/variants/training_length/nnUNetTrainer_E1.py')"], text=True).strip().splitlines()[-1])
if trainer_path.exists():
    require(trainer_path.read_text() == EMBEDDED["trainers.py"], "Eğitici kodu değişmiş; ortamı ve deney kimliğini kontrol et.")
else:
    trainer_path.write_text(EMBEDDED["trainers.py"])
# Keep all nnU-Net commands in the same external Python environment.
ENV = os.environ.copy()
ENV.update({"nnUNet_raw": str(DATA / "nnUNet_raw"), "nnUNet_preprocessed": str(PREPROCESSED),
            "nnUNet_results": str(RESULTS), "nnUNet_n_proc_DA": str(AUGMENTATION_WORKERS),
            "nnUNet_compile": "false", "PYTHONHASHSEED": "42",
            "MPLBACKEND": "Agg",  # External venv cannot use Colab's inline backend.
            "OMP_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1", "MKL_NUM_THREADS": "1"})
require(subprocess.check_output([str(PYTHON), "-c", "import sys; print(f'{sys.version_info.major}.{sys.version_info.minor}')"], text=True).strip() == "3.11", "Python sürümü 3.11 olmalı.")
gpu = json.loads(subprocess.check_output([str(PYTHON), "-c",
    "import sys,json; sys.path.insert(0, '/content/e1_tools'); from colab_helpers import gpu_probe; print(json.dumps(gpu_probe()))"], env=ENV, text=True))
print(json.dumps(gpu, indent=2))
session_id = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
(RESULTS / f"session_{session_id}.json").write_text(json.dumps({**gpu, "cu_per_hour": CU_PER_HOUR, "available_cu": AVAILABLE_CU}, indent=2))
(RESULTS / f"environment_{session_id}.txt").write_text(subprocess.check_output(uv + ["pip", "freeze", "--python", str(PYTHON)], text=True))


## 3 · Arşiv ve 54 MRI/etiket çiftini doğrula
Paket yalnız eğitim/doğrulama içerir; test klasörü kabul edilmez. Dosya sağlama toplamları,
44/10 ayrımı, voxel değerleri, şekiller, affine eşleşmesi ve tam kopya görüntüler kontrol edilir.
Bu teknik kontrol doku isimlerinin/anatomik etiketlerin uzman onayı değildir.


In [ ]:
extract_handoff(ARCHIVE, DATA)
stamp = str(time.time_ns())
run_logged([str(PYTHON), str(TOOLS / "audit_handoff.py"), str(DATA), "--out", str(WORK / "audit_report.json")],
           RESULTS / "logs" / f"audit_{stamp}.log", ENV)
report = json.loads((WORK / "audit_report.json").read_text())
verify_manifest(DATA, report)
require(report["dataset_sha256"] == EXPECTED_DATASET_SHA256 and report["split_sha256"] == EXPECTED_SPLIT_SHA256,
        "Arşiv, yerelde kontrol edilen teslim paketiyle aynı değil.")
require(report["train_count"] == 44 and report["validation_count"] == 10, "Bölme değişmiş.")
shutil.copyfile(WORK / "audit_report.json", RESULTS / f"audit_{stamp}.json")
unknown = sum("unknown" in case["header_spatial_units"] for case in report["cases"])
print(f"54 çift kontrol edildi: 44 eğitim / 10 doğrulama. Birimi belirtilmemiş vaka: {unknown}")
print("EĞİTİM ÖNCESİ NOTLAR:\n" + "\n".join(report["warnings"][:4]))


In [ ]:
# Uses training cases only; these are voxel-plane previews, not mm-space validation.
preview = RESULTS / "label_review.png"
subprocess.run([str(PYTHON), str(TOOLS / "preview_handoff.py"), str(DATA), "--out", str(preview)], check=True, env=ENV)
from IPython.display import display, Image
display(Image(filename=str(preview)))


**Veri inceleme noktası:** teslim notuna göre etiket dönüşümü çıkarımla yapılmış;
hava boşlukları arka plana birleştirilmiş; silinen yüz için `ignore` maskesi yok.
Bazı NYU affine'leri değiştirilmiş ve özgün dönüşüm manifesti pakette yok.
Başlıklarda fiziksel birim de belirtilmemiş. Bu çalışmadan hasta yüzü veya mm-doğruluğu iddiası üretme.

Görüntüleri ve bu sınırlamaları inceleyip ön deneye devam etmeyi seçersen ilk hücrede
`ACKNOWLEDGE_PROVISIONAL_DATA = True` yap. Bu, otomatik anatomik onay anlamına gelmez.


## 4 · Ön işleme ve sabit fold 0
Teslim edilen plan raporu: 1 mm, patch 128×160×112, batch 2. Burada kendi ortamımızın ürettiği plan gösterilir;
farklıysa eğitime geçmeden incele. Colab yeniden başlarsa yerel ön işleme yeniden yapılır;
Drive'daki öğrenilmiş ağırlıklar silinmez.


In [ ]:
require(ACKNOWLEDGE_PROVISIONAL_DATA, "Önizlemeleri/veri notlarını incele; sonra veri inceleme seçimini aç.")
marker = PREPROCESSED / ".e1_input_sha256"
preprocessing_key = hashlib.sha256((report["dataset_sha256"] + LOCK_SHA256).encode()).hexdigest()
if marker.is_file():
    require(marker.read_text().strip() == preprocessing_key, "Ön işleme başka veri/sürüme ait; yeni yerel klasör kullan.")
else:
    run_logged([str(VENV / "bin/nnUNetv2_plan_and_preprocess"), "-d", "1", "-c", "3d_fullres",
                "--verify_dataset_integrity", "-np", "1"], RESULTS / "logs" / f"preprocess_{time.time_ns()}.log", ENV)
    marker.parent.mkdir(parents=True, exist_ok=True)
    marker.write_text(preprocessing_key + "\n")
split_hash = install_split(DATA, PREPROCESSED)
require(split_hash == report["split_sha256"], "Sabit bölme kopyası değişmiş.")
plan_path = PREPROCESSED / DATASET / "nnUNetPlans.json"
plans = json.loads(plan_path.read_text())
require("3d_fullres" in plans["configurations"], "3d_fullres planı oluşmadı.")
print(json.dumps(plans["configurations"]["3d_fullres"], indent=2))
contract = {"experiment": "E1", "dataset_sha256": report["dataset_sha256"],
            "split_sha256": split_hash, "plans_sha256": sha256(plan_path), "lock_sha256": LOCK_SHA256,
            "trainer_sha256": TRAINER_SHA256, "configuration": "3d_fullres", "fold": 0,
            "nnunetv2": "2.8.1", "torch": "2.6.0+cu124", "seed": 42,
            "augmentation_workers": AUGMENTATION_WORKERS, "torch_compile": False,
            "preliminary_only": True, "test_accessed": False}
record_run(RESULTS / "experiment_contract.json", contract)
for path in (plan_path, PREPROCESSED / DATASET / "dataset_fingerprint.json", DATA / "splits_final.json"):
    target = RESULTS / path.name
    if target.exists():
        require(sha256(target) == sha256(path), f"Kayıtlı {path.name} değişmiş; yeni deney kimliği kullan.")
    else:
        shutil.copyfile(path, target)


## 5 · Yalnız 5 epoch kontrol çalışması
`RUN_5_EPOCH_PILOT = True` seçimi ücretli GPU çalışmasını başlatır.
Her tamamlanan epoch'ta checkpoint alınır. Bir epoch tamamlanmadan koparsa başlangıç ağırlıkları kaydedilmiş olmayabilir.
Devam için aynı deney kimliği + `RESUME_PILOT = True`; checkpoint yoksa sessiz yeniden başlama yapılmaz.
Başlangıç seed'i 42; CUDA/çok işçili artırmalar bit düzeyinde tekrarlanabilirlik garantisi vermez.


In [ ]:
def train_stage(trainer, enabled, resume):
    if not enabled:
        print(f"{trainer}: kapalı; eğitim başlatılmadı.")
        return
    require(ACKNOWLEDGE_PROVISIONAL_DATA and (RESULTS / "experiment_contract.json").is_file(), "Veri/ön işleme adımları tamamlanmalı.")
    require(os.path.ismount("/content/drive"), "Checkpoint hedefi kalıcı Drive bağlantısında olmalı.")
    require(ENV["nnUNet_results"] == str(RESULTS) and ENV["nnUNet_raw"] == str(DATA / "nnUNet_raw")
            and ENV["nnUNet_n_proc_DA"] == str(AUGMENTATION_WORKERS), "Ayarlar değişti; ortam/veri/ön işleme hücrelerini yeniden çalıştır.")
    record_run(RESULTS / "experiment_contract.json", contract)
    folder = RESULTS / DATASET / f"{trainer}__nnUNetPlans__3d_fullres" / "fold_0"
    command = training_command(VENV / "bin/nnUNetv2_train", trainer, folder, resume)
    log = RESULTS / "logs" / f"{trainer}_{time.time_ns()}.log"
    result = run_logged(command, log, ENV)
    (log.with_suffix(".json")).write_text(json.dumps(result, indent=2))
    print(f"Bitti: {result['elapsed_seconds']/60:.1f} dakika. Log: {log}")
train_stage("nnUNetTrainer_E1Smoke", RUN_5_EPOCH_PILOT, RESUME_PILOT)


## 6 · Süre/birim tahmini
İlk epoch'un başlangıç maliyetini dışarıda bırakıp sonraki epoch sürelerinin medyanı kullanılır.
Hesap yalnız eğitim döngüsüne yaklaşır; kurulum, ön işleme, nihai doğrulama ve değişen hız için ek pay gerekir.
1.23 birim/saat bilgisini başka GPU için kullanma; açık oturumun kendi oranını gir.


In [ ]:
logs = sorted((RESULTS / "logs").glob("nnUNetTrainer_E1Smoke_*.log"))
epoch_seconds = []
for log in logs:
    epoch_seconds.extend(float(value) for value in re.findall(r"Epoch time:\s*([0-9.]+)", log.read_text()))
if len(epoch_seconds) >= 2:
    import statistics
    median = statistics.median(epoch_seconds[1:])
    hours = median * 100 / 3600
    print(f"100 epoch eğitim döngüsü yaklaşık {hours:.2f} saat; ek aşamalar dahil değil.")
    if CU_PER_HOUR > 0:
        units = hours * CU_PER_HOUR
        print(f"Girilen oranla yaklaşık {units:.1f} işlem birimi + ek aşamalar.")
        if AVAILABLE_CU > 0:
            print(f"Girilen hesap bakiyesi: {AVAILABLE_CU:.1f}. Bu hesap yeterlilik garantisi değildir.")
else:
    print("Tahmin için en az iki tamamlanmış epoch süresi gerekli.")


## 7 · Ayrı 100 epoch ön model — varsayılan kapalı
Kontrol çalışmasının hızını, görüntülerini ve bütçeyi inceledikten sonra
`RUN_100_EPOCH_PRELIMINARY = True` yap. Bu eğitici ayrı klasörde **sıfırdan** başlar;
5 epoch ağırlıklarını otomatik aktarmaz. Kendi kesilmiş 100 epoch çalışmasına devam için
`RESUME_PRELIMINARY = True` kullan. Mevcut sonuçların üzerine yeni eğitim yazılmaz.

**100 epoch bir ön sonuçtur**, nihai proje tamamlanması/test başarısı değildir.
Ücretli uzun çalıştırmayı burada otomatik açma.


In [ ]:
train_stage("nnUNetTrainer_E1Preliminary", RUN_100_EPOCH_PRELIMINARY, RESUME_PRELIMINARY)


## 8 · Doğrulama çıktıları
Epoch logundaki patch tabanlı pseudo-Dice ile tüm 10 doğrulama vakasının son segmentasyon Dice'ı farklıdır.
Nihai doğrulama bitmediyse sonuç varmış gibi raporlanmaz. Test kümesi bu defterde açılmaz.


In [ ]:
summary = RESULTS / DATASET / "nnUNetTrainer_E1Preliminary__nnUNetPlans__3d_fullres/fold_0/validation/summary.json"
if summary.is_file():
    metrics = json.loads(summary.read_text())
    print("Yalnız 10 doğrulama vakası — ön sonuç:")
    print(json.dumps({"foreground_mean": metrics.get("foreground_mean"), "per_label_mean": metrics.get("mean")}, indent=2))
else:
    print("Tamamlanmış 100 epoch doğrulama özeti henüz yok.")
print("Ara verirken: checkpoint/logların Drive'da bulunduğunu kontrol et; kullanılmayan GPU oturumunu sonlandır.")


Kaynaklar: [nnU-Net v2.8.1](https://github.com/MIC-DKFZ/nnUNet/tree/v2.8.1),
[5/100 epoch eğiticileri](https://github.com/MIC-DKFZ/nnUNet/blob/v2.8.1/nnunetv2/training/nnUNetTrainer/variants/training_length/nnUNetTrainer_Xepochs.py),
[Colab kaynak sınırları](https://research.google.com/colaboratory/faq.html).
CHARM E2 için ayrı etiketleme aracı olabilir; bu E1 defteri yalnız teslim edilen MultiAxial verisini kullanır.
